# Fine-Tuning a Transformer with LoRA (Google Colab)

We **fine-tune** DistilBERT to classify clinical notes into 3 types, using
**LoRA** (a cheap, efficient fine-tuning method). This is real deep-learning
training - it needs a GPU, so run it in **Google Colab**.

**How to run:**
1. Go to https://colab.research.google.com  ->  File -> Upload notebook -> pick this file.
2. Menu: **Runtime -> Change runtime type -> Hardware accelerator: GPU -> Save**.
3. Run each cell top to bottom (Shift+Enter).

**What you'll learn:** tokens, epochs, loss, learning rate, LoRA, transfer
learning, trainable parameters.

## Recap: what is fine-tuning + LoRA?

- **Pretraining** (done by others): the model already learned general English.
- **Fine-tuning** (us): train it a bit more on OUR labeled data so it learns OUR
  task. This is **transfer learning**.
- **LoRA** (Low-Rank Adaptation): instead of updating ALL the model's millions of
  weights, we FREEZE them and train tiny added "adapter" weights (~1%). Fast,
  cheap, runs on a free GPU. (**QLoRA** = LoRA + 4-bit compression for even bigger
  models.)

In [ ]:
# Cell 1 - install the libraries (Colab)
!pip -q install transformers datasets peft accelerate

In [ ]:
# Cell 2 - imports + check the GPU is on
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from peft import LoraConfig, get_peft_model, TaskType

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "(should say 'cuda' - if 'cpu', enable GPU in Runtime settings)")

## Step 1 - the labeled data (our task)

Each example = clinical text + its type (0/1/2). Real projects use thousands of
de-identified documents; here we use a small set to keep it fast.

In [ ]:
# Cell 3 - dataset
label_names = ["Physician Note", "Discharge Summary", "Lab Report"]

data = [
 ("The patient presents with hypertension, BP 150/95.", 0),
 ("Assessment: type 2 diabetes on metformin, continue plan.", 0),
 ("Chief complaint: headaches, start medication, follow up.", 0),
 ("Physical exam unremarkable, reports chest discomfort.", 0),
 ("Plan: reduce salt, exercise, recheck in four weeks.", 0),
 ("Assessment: mild asthma, prescribe inhaler, review monthly.", 0),
 ("Complaint of back pain, advised physiotherapy and rest.", 0),
 ("Follow-up care: finish antibiotics, see doctor in a week.", 1),
 ("Hospital course: pneumonia, IV antibiotics, improved.", 1),
 ("Discharge instructions: return to ER if fever returns.", 1),
 ("Patient discharged in stable condition after three days.", 1),
 ("Reason for admission: pneumonia, discharged home.", 1),
 ("Discharged with wound care instructions and follow-up.", 1),
 ("Post-op discharge: rest, no lifting, review in ten days.", 1),
 ("Fasting glucose 138 high, cholesterol 220 borderline.", 2),
 ("Results: LDL 145, HDL 40, repeat testing advised.", 2),
 ("Metabolic panel: elevated glucose, lipid profile abnormal.", 2),
 ("Hemoglobin A1c 7.8 percent, above target range.", 2),
 ("Panel: sodium 140, potassium 4.2, creatinine normal.", 2),
 ("Blood count shows mild anemia, hemoglobin 10.5.", 2),
 ("Thyroid panel: TSH elevated, free T4 low.", 2),
]

texts  = [t for t, y in data]
labels = [y for t, y in data]

# simple train/test split (last 2 of each class as test)
test_idx = [5,6, 12,13, 19,20]
train = [(t,y) for i,(t,y) in enumerate(data) if i not in test_idx]
test  = [(t,y) for i,(t,y) in enumerate(data) if i in test_idx]
print("train:", len(train), " test:", len(test))

## Step 2 - tokenizer (text -> tokens -> numbers)

The model reads numbers, not words. The tokenizer splits text into tokens and
maps them to IDs (same idea we saw earlier).

In [ ]:
# Cell 4 - tokenizer + build tokenized datasets
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def make_ds(rows):
    ds = Dataset.from_dict({"text":[t for t,y in rows], "labels":[y for t,y in rows]})
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True), batched=True)
    return ds.remove_columns(["text"])

train_ds = make_ds(train)
test_ds  = make_ds(test)
print(train_ds)

## Step 3 - load the model + attach LoRA

We load DistilBERT with a 3-class classification head, then wrap it with LoRA.
`print_trainable_parameters()` shows LoRA trains only a TINY fraction of weights.

LoRA settings:
- `r=8` = size (rank) of the small adapters (bigger r = more capacity).
- `lora_alpha=16` = a scaling factor for the adapters.
- `target_modules` = which layers get adapters (the attention query/value).

In [ ]:
# Cell 5 - model + LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3,
    id2label={i:n for i,n in enumerate(label_names)},
    label2id={n:i for i,n in enumerate(label_names)},
)

lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
                  lora_dropout=0.1, target_modules=["q_lin","v_lin"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # <-- see how few weights LoRA trains

## Step 4 - training settings (the terms you asked about)

- **epoch** = one full pass over the training data. `num_train_epochs=10` = 10 passes.
- **batch size** = how many examples the model looks at before each weight update.
- **learning rate** = how big each weight update is (too big = unstable, too small = slow).
- **loss** = how wrong the model is; training pushes it DOWN each step.

In [ ]:
# Cell 6 - train (fine-tune)
args = TrainingArguments(
    output_dir="out",
    num_train_epochs=10,            # 10 passes over the data
    per_device_train_batch_size=4,  # 4 examples per step
    learning_rate=2e-4,             # step size
    logging_steps=5,                # print loss every 5 steps
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
)
trainer.train()   # watch the 'loss' column go DOWN = learning

## Step 5 - evaluate + predict

Check accuracy on the held-out test set, then predict on brand-new text.

In [ ]:
# Cell 7 - accuracy on test set + predict new text
import numpy as np

def predict(texts):
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
    ids = logits.argmax(-1).tolist()
    return [label_names[i] for i in ids]

# test accuracy
test_texts  = [t for t,y in test]
test_labels = [y for t,y in test]
preds = predict(test_texts)
acc = np.mean([label_names[y]==p for y,p in zip(test_labels, preds)])
print("Test accuracy:", round(float(acc),2))
print()

# predict new sentences
for t in ["Blood pressure 160/100, start medication, follow up.",
          "Discharged home with antibiotics, return if worse.",
          "Glucose 150 high, cholesterol elevated, repeat labs."]:
    print(f"{predict([t])[0]:<18} <- {t}")

## Summary + how production fine-tuning scales

You just:
- **fine-tuned** a real transformer (DistilBERT) with **LoRA** on a GPU,
- trained only ~1% of the weights (see the trainable-parameters line),
- watched the **loss** fall over **epochs**, and used the model to **predict**.

**In production** you would: use thousands of labeled examples, tune epochs /
learning rate / LoRA `r`, evaluate with precision/recall/F1, **save the LoRA
adapter** (`model.save_pretrained("adapter")`) - it's tiny (a few MB) - and load
it on top of the base model to serve. For bigger models, use **QLoRA** (4-bit)
to fit them on one GPU.

Interview soundbite: "I fine-tuned with LoRA/PEFT, training a small set of
adapter weights instead of the full model - same accuracy, a fraction of the
compute, and the adapter is a few MB to ship." 